
# **Deep Learning for NLP : IndustryGPT: Specialized LLM Bot Using Pre-Trained Models**

CONTRIBUTION - INDIVIDUAL

GITHUB -

## Introduction
The rapid advancement of Artificial Intelligence and Natural Language Processing (NLP) has transformed customer interaction systems across industries, especially in banking. Traditional rule-based chatbots often fail to understand natural language queries and provide context-aware responses, leading to inefficient customer support.

This project focuses on developing an AI-powered Retail Banking Chatbot using a fine-tuned Large Language Model (LLM). By leveraging the FLAN-T5 transformer model from Hugging Face and training it on banking-related conversational data, the chatbot can respond intelligently to customer queries related to account services, debit cards, loans, and transactions.

The project demonstrates the practical application of deep learning and NLP techniques in automating customer support within the retail banking industry. A Gradio-based interface is also developed to enable real-time interaction between users and the chatbot.

---

## Project Statement
The retail banking industry handles a large volume of customer queries related to banking services, where traditional rule-based chatbots often fail to provide accurate and context-aware responses. This project aims to develop an AI-powered retail banking chatbot using a fine-tuned Large Language Model (LLM) capable of understanding and responding to real customer queries effectively. By leveraging the FLAN-T5-small model and banking-specific conversational data, the chatbot provides fast, informative, and human-like responses through an interactive Gradio-based interface.

---



## Project Objectives

- To develop an intelligent retail banking chatbot using a fine-tuned Large Language Model (LLM).
- To enable the chatbot to accurately understand and process banking-related customer queries.
- To utilize banking-specific conversational datasets for training and improving response generation.
- To automate customer support services related to account management, card services, loans, and transactions.
- To implement Natural Language Processing (NLP) techniques for generating context-aware and human-like responses.
- To design and deploy an interactive chatbot interface using Gradio for real-time user communication.
- To demonstrate the practical application of transformer-based deep learning models in enhancing digital banking support systems.

---



## Project Summary

The “AI-Powered Retail Banking Chatbot Using Fine-Tuned Large Language Models” project aims to develop an intelligent conversational assistant capable of handling banking-related customer queries effectively. The chatbot is built using the FLAN-T5 Small model from Hugging Face and trained on the Bitext Retail Banking Dataset containing real-world banking conversations.

The project involves data preprocessing, model fine-tuning, and chatbot deployment using Gradio. The trained model learns banking-specific language patterns and generates context-aware responses for queries related to account services, debit cards, loans, and customer support operations.

This project demonstrates how Large Language Models and Natural Language Processing techniques can be used to automate customer support services in the retail banking industry, improving response efficiency and enhancing user experience.

---


In [21]:
# The pip install command was used to install all required Python libraries such as Transformers, Datasets, Scikit-learn, and Gradio,
# which are necessary for model training, dataset handling, and chatbot deployment.

!pip install transformers datasets scikit-learn gradio


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("/content/drive/My Drive/Masters in CS:AI and ML/CS_AI_M3_Deep_Learning_for_NLP:_LLM/bitext-retail-banking-llm-chatbot-training-dataset.csv")

# Data Preprocessing
# Keep only needed columns and drop missing values
df_clean = df[['instruction', 'response']].dropna()
df_clean = df_clean[df_clean['instruction'].str.strip() != ""]
df_clean = df_clean[df_clean['response'].str.strip() != ""]
df_clean = df_clean.drop_duplicates(subset=['instruction', 'response'])

# Rename for model input/output
df_clean = df_clean.rename(columns={"instruction": "input", "response": "output"})

# Preview
df_clean.sample(3)


,input,output
2963,can ya help me apply for an loan,Definitely! I'd be happy to help you apply for...
5153,"I need to cancel a credit card on mobile, how ...",You bet! I'm here to assist you with canceling...
21597,whereto get a password,Without a doubt! I can assist you with that. T...


In [24]:
# Train-test split here 90-10 split is used

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_clean, test_size=0.1, random_state=42)


FLAN-T5 is a pre-trained Large Language Model (LLM) developed by Google for Natural Language Processing (NLP) tasks such as question answering, summarization, translation, and chatbot response generation.

FLAN-T5 is a lightweight transformer-based sequence-to-sequence model optimized for instruction-following tasks. It is suitable for chatbot applications and efficient for training on Google Colab

Now, below code loads the pre-trained FLAN-T5-small transformer model and its tokenizer from the Hugging Face platform using the Transformers library. The tokenizer is responsible for converting human-readable text into token IDs, while the model is used for generating chatbot responses. During execution, the required model files such as configuration files, tokenizer files, and model weights are automatically downloaded and loaded into the Google Colab environment for fine-tuning and inference.

In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Here code converts the training and testing datasets into Hugging Face dataset format and tokenizes the text data so the FLAN-T5 model can understand it. The tokenizer converts customer questions and chatbot responses into numerical token IDs, while also creating labels for model training. Finally, the .map() function applies tokenization to the entire dataset before training begins.

In [ ]:
from datasets import Dataset

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize(example):
    input_enc = tokenizer(example["input"], padding="max_length", truncation=True, max_length=128)
    target_enc = tokenizer(example["output"], padding="max_length", truncation=True, max_length=128)
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

# Tokenize datasets
train_tokenized = train_dataset.map(tokenize)
test_tokenized = test_dataset.map(tokenize)


Map:   0%|          | 0/22990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2555 [00:00<?, ? examples/s]

Below code configures the training settings for the FLAN-T5 model using Hugging Face’s Seq2SeqTrainingArguments, including batch size, number of epochs, logging, and evaluation strategy. The DataCollatorForSeq2Seq handles proper batching and padding of tokenized data, while Seq2SeqTrainer initializes the complete training pipeline by connecting the model, datasets, tokenizer, and training configurations together before model training begins.

In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./banking_bot",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    num_train_epochs=3,
    logging_dir='./logs',
    eval_strategy="epoch"
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=collator
)

trainer.train()

Below code is used to evaluate the trained FLAN-T5 model by extracting the training loss and validation loss after model training. The trainer.evaluate() function tests the model on unseen test data and provides evaluation metrics such as evaluation loss, runtime, and processing speed. Finally, the trained model and tokenizer are saved using save_pretrained() so they can be reused later without retraining the model.

In [ ]:
# prompt: give testing accuracy training accuracy and validation accuracy

# After trainer.train()
import json

# Function to extract metrics from logs
def get_metrics(log_history):
    train_loss = None
    eval_loss = None
    # Find the final training loss
    for log in log_history:
        if 'loss' in log:
            train_loss = log['loss']
        if 'eval_loss' in log:
            eval_loss = log['eval_loss']
    return train_loss, eval_loss

train_loss, eval_loss = get_metrics(trainer.state.log_history)

print(f"Training Loss (last logged): {train_loss}")
print(f"Validation Loss (last epoch): {eval_loss}")

# To get evaluation metrics on the test set after training
eval_results = trainer.evaluate(test_tokenized)
print(f"Test Evaluation Results: {eval_results}")

# Note: For accuracy in seq2seq models, it's usually not a single number
# like classification accuracy. Metrics like ROUGE, BLEU are common.
# If you need specific accuracy-like metrics, you would need to implement
# or use a library for sequence generation evaluation metrics and potentially
# override the compute_metrics function in the Trainer or perform manual evaluation.
# The default evaluation in Seq2SeqTrainer provides loss.



In [ ]:
model.save_pretrained("banking-chatbot")
tokenizer.save_pretrained("banking-chatbot")


Below code defines the chatbot function responsible for generating responses using the trained FLAN-T5 model. The user’s banking-related query is first tokenized and converted into tensor format so that it can be processed by the transformer model. The tensors are then moved to the appropriate device, such as GPU or CPU, to ensure efficient execution. Using the model.generate() function, the FLAN-T5 model generates a context-aware banking response based on the user input. Finally, the generated token output is decoded back into human-readable text and displayed as the chatbot’s response.

This code is used to implement the core response generation mechanism of the Retail Banking Chatbot. It enables real-time interaction between users and the chatbot by processing natural language queries and generating meaningful responses dynamically. The function also ensures optimized execution by utilizing available hardware resources and demonstrates the practical application of transformer-based Large Language Models in automating customer support services within the banking industry.

In [ ]:
import torch

def chat(prompt):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Tokenize and move tensors to the same device as model
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate response
    with torch.no_grad():
        output = model.generate(**inputs, max_length=100)

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
chat("How can I activate my debit card?")


Gradio is an open-source Python library used to create interactive web interfaces for Machine Learning and AI applications. In this project, Gradio is used to develop and deploy the Retail Banking Chatbot as a user-friendly web application. It provides input and output textboxes that allow users to enter banking-related queries and receive AI-generated responses in real time. The gr.Interface() function connects the chatbot model with the web interface, while interface.launch() generates a public URL for live interaction through a browser.

Below code creates a Gradio-based web interface for the Retail Banking Chatbot. The gr.Interface() function connects the chatbot function (chat) with input and output textboxes, allowing users to enter banking-related queries and receive generated responses in real time. The interface also includes a title for the chatbot application, making it user-friendly and interactive.

In [ ]:
import gradio as gr

In [ ]:
interface = gr.Interface(
    fn=chat,          # The function to wrap with a UI
    inputs=gr.Textbox(label="Your banking question"), # Input component: a textbox for the user's prompt
    outputs=gr.Textbox(label="Chatbot response"), # Output component: a textbox for the chatbot's response
    title="Retail Banking Chatbot" # Title for the Gradio interface
)

Below code launches the Gradio-based chatbot interface using interface.launch(), which deploys the Retail Banking Chatbot as an interactive web application. It generates a public URL that allows users to access the chatbot and interact with it in real time through a browser-based interface.

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Your banking question"),
    outputs=gr.Textbox(label="Chatbot response"),
    title="Retail Banking Chatbot"
)

interface.launch()